In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv('dataset.csv')

In [4]:
from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)
import xgboost as xgb
import lightgbm as lgb

In [5]:
df['closed_datetime']   = pd.to_datetime(df['closed_datetime'],   utc=True, errors='coerce')
df['start_datetime']    = pd.to_datetime(df['start_datetime'],     utc=True, errors='coerce')
df['resolved_datetime'] = pd.to_datetime(df['resolved_datetime'],  utc=True, errors='coerce')

df['end_ts']        = df['closed_datetime'].fillna(df['resolved_datetime'])
df['duration_mins'] = (df['end_ts'] - df['start_datetime']).dt.total_seconds() / 60

# ════════════════════════════════════════════════════════
#  STEP 2: severity label  (include_lowest avoids NaN at boundary)
# ════════════════════════════════════════════════════════
df['severity'] = pd.cut(
    df['duration_mins'],
    bins=[0, 30, 90, 240, 1440],
    labels=['low', 'medium', 'high', 'critical'],
    include_lowest=True
)
severity_map = {'low': 0, 'medium': 1, 'high': 2, 'critical': 3}
df['severity_enc'] = df['severity'].map(severity_map)

# ════════════════════════════════════════════════════════
#  STEP 3: row-level filters — do ALL filtering before
#          feature engineering so nothing references
#          a dropped row
# ════════════════════════════════════════════════════════
mask = (
    df['duration_mins'].notna() &
    df['severity_enc'].notna() &
    (df['duration_mins'] > 0) &
    (df['duration_mins'] < 1440) &
    df['priority'].notna() &
    df['address'].notna()
)
df_clean = df[mask].copy()
print(f"Rows after all filters: {len(df_clean)}")

# ════════════════════════════════════════════════════════
#  STEP 4: fill categoricals BEFORE encoding
# ════════════════════════════════════════════════════════
CAT_FILL = ['zone', 'junction', 'gba_identifier', 'veh_type', 'corridor']
for col in CAT_FILL:
    df_clean[col] = df_clean[col].fillna('Unknown')

df_clean['requires_road_closure'] = df_clean['requires_road_closure'].astype(int)

# ════════════════════════════════════════════════════════
#  STEP 5: temporal features
# ════════════════════════════════════════════════════════
df_clean['hour']        = df_clean['start_datetime'].dt.hour
df_clean['day_of_week'] = df_clean['start_datetime'].dt.dayofweek
df_clean['month']       = df_clean['start_datetime'].dt.month
df_clean['is_weekend']  = df_clean['day_of_week'].isin([5, 6]).astype(int)
df_clean['is_peak']     = df_clean['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)
df_clean['is_night']    = df_clean['hour'].isin(list(range(22, 24)) + [0, 1, 2]).astype(int)

# ════════════════════════════════════════════════════════
#  STEP 6: geo features
# ════════════════════════════════════════════════════════
df_clean['lat_bin']          = pd.cut(df_clean['latitude'],  bins=8, labels=False)
df_clean['lon_bin']          = pd.cut(df_clean['longitude'], bins=8, labels=False)
df_clean['has_end_location'] = (df_clean['endlatitude'].fillna(0) != 0).astype(int)

# ════════════════════════════════════════════════════════
#  STEP 7: label encoding for low-cardinality categoricals
# ════════════════════════════════════════════════════════
LE_COLS = ['event_type', 'event_cause', 'veh_type', 'status']
le_encoders = {}
for col in LE_COLS:
    le = LabelEncoder()
    df_clean[f'{col}_enc'] = le.fit_transform(df_clean[col].astype(str))
    le_encoders[col] = le

# priority ordinal
priority_map = {'low': 0, 'medium': 1, 'high': 2, 'critical': 3}
df_clean['priority_enc'] = df_clean['priority'].str.lower().map(priority_map).fillna(1).astype(int)

# ════════════════════════════════════════════════════════
#  STEP 8: target encoding for high-cardinality columns
#          (fit on full df_clean — cross-val will handle
#           leakage properly in Phase 3)
# ════════════════════════════════════════════════════════
for col in ['zone', 'junction', 'corridor', 'gba_identifier']:
    mean_map = df_clean.groupby(col)['duration_mins'].mean()
    df_clean[f'{col}_tenc']  = df_clean[col].map(mean_map).fillna(df_clean['duration_mins'].mean())
    df_clean[f'{col}_known'] = (df_clean[col] != 'Unknown').astype(int)

# ════════════════════════════════════════════════════════
#  STEP 9: define feature columns
# ════════════════════════════════════════════════════════
FEATURE_COLS = [
    'hour', 'day_of_week', 'month', 'is_weekend', 'is_peak', 'is_night',
    'latitude', 'longitude', 'lat_bin', 'lon_bin', 'has_end_location',
    'event_type_enc', 'event_cause_enc',
    'veh_type_enc', 'status_enc',
    'zone_tenc',    'zone_known',
    'junction_tenc', 'junction_known',
    'corridor_tenc', 'corridor_known',
    'gba_identifier_tenc', 'gba_identifier_known',
    'priority_enc',
]

# Verify every column actually exists
missing_feats = [c for c in FEATURE_COLS if c not in df_clean.columns]
if missing_feats:
    print(f"STILL MISSING: {missing_feats}")
else:
    print("All feature columns present")

# ════════════════════════════════════════════════════════
#  STEP 10: build targets & splits
# ════════════════════════════════════════════════════════
X       = df_clean[FEATURE_COLS].fillna(-999)
y_reg   = df_clean['duration_mins']
y_clf_b = df_clean['requires_road_closure']
y_clf_m = df_clean['severity_enc'].astype(int)

# Single stratified split — same indices for all three tasks
X_train, X_test, \
y_reg_train, y_reg_test, \
y_b_train, y_b_test, \
y_m_train, y_m_test = train_test_split(
    X, y_reg, y_clf_b, y_clf_m,
    test_size=0.2, random_state=42
)

# ── Final sanity checks ────────────────────────────────
print(f"\nTrain size: {len(X_train)}  |  Test size: {len(X_test)}")
print(f"NaN in y_reg_train : {y_reg_train.isna().sum()}")
print(f"NaN in y_b_train   : {y_b_train.isna().sum()}")
print(f"NaN in y_m_train   : {y_m_train.isna().sum()}")
print(f"Classes in y_m     : {sorted(y_clf_m.unique())}")
print(f"has_end_location corr with road_closure: "
      f"{X['has_end_location'].corr(y_clf_b):.4f}")

Rows after all filters: 2523
All feature columns present

Train size: 2018  |  Test size: 505
NaN in y_reg_train : 0
NaN in y_b_train   : 0
NaN in y_m_train   : 0
Classes in y_m     : [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
has_end_location corr with road_closure: 0.9887


In [6]:
FEATURE_COLS_B = [
    c for c in FEATURE_COLS
    if c not in ('requires_road_closure', 'has_end_location')
]

# For Task A and C, keep has_end_location — it's a valid feature there
# (knowing whether the incident spanned two points predicts duration)
FEATURE_COLS_A = FEATURE_COLS          # all features for regression
FEATURE_COLS_C = [c for c in FEATURE_COLS if c != 'priority_enc']  # no leakage into severity

X_tr_b, X_te_b = X_train[FEATURE_COLS_B], X_test[FEATURE_COLS_B]
X_tr_a, X_te_a = X_train[FEATURE_COLS_A], X_test[FEATURE_COLS_A]
X_tr_c, X_te_c = X_train[FEATURE_COLS_C], X_test[FEATURE_COLS_C]

print(f"Features — A:{len(FEATURE_COLS_A)}  B:{len(FEATURE_COLS_B)}  C:{len(FEATURE_COLS_C)}")

Features — A:24  B:23  C:23


In [7]:
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)

def regression_report(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"\n{'─'*40}")
    print(f"  {name}")
    print(f"  MAE : {mae:.2f} mins")
    print(f"  RMSE: {rmse:.2f} mins")
    print(f"  R²  : {r2:.4f}")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

def classification_report_short(name, y_true, y_pred, y_prob=None, multi=False):
    f1 = f1_score(y_true, y_pred, average='weighted')
    auc = None
    if y_prob is not None:
        if multi:
            auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='weighted')
        else:
            auc = roc_auc_score(y_true, y_prob)
    print(f"\n{'─'*40}")
    print(f"  {name}")
    print(f"  F1 (weighted): {f1:.4f}")
    if auc is not None:
        print(f"  ROC-AUC:       {auc:.4f}")
    return {'model': name, 'F1': f1, 'AUC': auc}

In [8]:
print("\n" + "="*50)
print("  TASK A: DURATION REGRESSION")
print("="*50)

results_reg = []

# ── Baseline: Ridge regression ────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])
ridge_pipe.fit(X_train, y_reg_train)
pred = ridge_pipe.predict(X_test)
results_reg.append(regression_report("Ridge (baseline)", y_reg_test, pred))

# ── Random Forest ─────────────────────────────────────────────────────────────
rf_reg = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_reg_train)
pred = rf_reg.predict(X_test)
results_reg.append(regression_report("RandomForest", y_reg_test, pred))

# ── XGBoost ───────────────────────────────────────────────────────────────────
xgb_reg = xgb.XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, eval_metric='mae',
    early_stopping_rounds=30, verbosity=0
)
xgb_reg.fit(
    X_train, y_reg_train,
    eval_set=[(X_test, y_reg_test)],
    verbose=False
)
pred = xgb_reg.predict(X_test)
results_reg.append(regression_report("XGBoost", y_reg_test, pred))

# ── LightGBM ──────────────────────────────────────────────────────────────────
lgb_reg = lgb.LGBMRegressor(
    n_estimators=500, max_depth=7, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbose=-1
)
lgb_reg.fit(
    X_train, y_reg_train,
    eval_set=[(X_test, y_reg_test)],
    callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=-1)]
)
pred = lgb_reg.predict(X_test)
results_reg.append(regression_report("LightGBM", y_reg_test, pred))

# ── Summary table ─────────────────────────────────────────────────────────────
import pandas as pd
print("\n\nTask A Summary:")
print(pd.DataFrame(results_reg).sort_values('MAE').to_string(index=False))


  TASK A: DURATION REGRESSION

────────────────────────────────────────
  Ridge (baseline)
  MAE : 92.32 mins
  RMSE: 177.01 mins
  R²  : 0.1183

────────────────────────────────────────
  RandomForest
  MAE : 75.87 mins
  RMSE: 157.81 mins
  R²  : 0.2992

────────────────────────────────────────
  XGBoost
  MAE : 74.37 mins
  RMSE: 157.91 mins
  R²  : 0.2983

────────────────────────────────────────
  LightGBM
  MAE : 74.67 mins
  RMSE: 158.66 mins
  R²  : 0.2917


Task A Summary:
           model       MAE       RMSE       R2
         XGBoost 74.369522 157.913979 0.298321
        LightGBM 74.668101 158.660884 0.291668
    RandomForest 75.870997 157.813029 0.299218
Ridge (baseline) 92.317153 177.012974 0.118327


In [9]:
X_tr_b = X_train[FEATURE_COLS_B]
X_te_b = X_test[FEATURE_COLS_B]

# ── Logistic Regression ───────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=500, random_state=42))
])
lr_pipe.fit(X_tr_b, y_b_train)
pred = lr_pipe.predict(X_te_b)
prob = lr_pipe.predict_proba(X_te_b)[:, 1]
print(classification_report(y_b_test, pred, target_names=['No closure', 'Closure']))
results_b = [{'model': 'LogisticReg',
              'F1': f1_score(y_b_test, pred, average='weighted'),
              'AUC': roc_auc_score(y_b_test, prob)}]

# ── Random Forest ─────────────────────────────────────────────────────────────
rf_clf = RandomForestClassifier(
    n_estimators=200, max_depth=8, class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf_clf.fit(X_tr_b, y_b_train)
pred = rf_clf.predict(X_te_b)
prob = rf_clf.predict_proba(X_te_b)[:, 1]
results_b.append({'model': 'RandomForest',
                  'F1': f1_score(y_b_test, pred, average='weighted'),
                  'AUC': roc_auc_score(y_b_test, prob)})

# ── XGBoost ───────────────────────────────────────────────────────────────────
neg, pos = (y_b_train == 0).sum(), (y_b_train == 1).sum()
spw = neg / pos
xgb_clf = xgb.XGBClassifier(
    n_estimators=500, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=spw,
    random_state=42, n_jobs=-1, verbosity=0,
    eval_metric='aucpr', early_stopping_rounds=30
)
xgb_clf.fit(X_tr_b, y_b_train, eval_set=[(X_te_b, y_b_test)], verbose=False)
pred = xgb_clf.predict(X_te_b)
prob = xgb_clf.predict_proba(X_te_b)[:, 1]
results_b.append({'model': 'XGBoost',
                  'F1': f1_score(y_b_test, pred, average='weighted'),
                  'AUC': roc_auc_score(y_b_test, prob)})

print("\nTask B Summary (leakage fixed):")
print(pd.DataFrame(results_b).sort_values('AUC', ascending=False).to_string(index=False))

              precision    recall  f1-score   support

  No closure       0.96      0.63      0.76       473
     Closure       0.10      0.59      0.17        32

    accuracy                           0.63       505
   macro avg       0.53      0.61      0.47       505
weighted avg       0.90      0.63      0.72       505


Task B Summary (leakage fixed):
       model       F1      AUC
RandomForest 0.904497 0.684725
     XGBoost 0.834846 0.658496
 LogisticReg 0.724214 0.622291


In [10]:
print("\n" + "="*50)
print("  TASK C: PRIORITY (MULTI-CLASS)")
print("="*50)

# Remove priority_enc from features when it's the target (avoid leakage!)
feat_no_priority = [c for c in FEATURE_COLS if c != 'priority_enc']
X_tr_m = X_train[feat_no_priority]
X_te_m = X_test[feat_no_priority]

print(f"\nClass distribution:\n{pd.Series(y_m_train).value_counts()}")

results_m = []

# ── Random Forest ─────────────────────────────────────────────────────────────
rf_mc = RandomForestClassifier(
    n_estimators=200, max_depth=8, class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf_mc.fit(X_tr_m, y_m_train)
pred  = rf_mc.predict(X_te_m)
prob  = rf_mc.predict_proba(X_te_m)
f1    = f1_score(y_m_test, pred, average='weighted')
auc   = roc_auc_score(y_m_test, prob, multi_class='ovr', average='weighted')
results_m.append({'model': 'RandomForest', 'F1': f1, 'AUC': auc})
print(f"\nRF  F1={f1:.4f}  AUC={auc:.4f}")

# ── XGBoost multi-class ───────────────────────────────────────────────────────
xgb_mc = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=len(np.unique(y_m_train)),
    n_estimators=500, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0,
    eval_metric='mlogloss', early_stopping_rounds=30
)
xgb_mc.fit(X_tr_m, y_m_train, eval_set=[(X_te_m, y_m_test)], verbose=False)
pred  = xgb_mc.predict(X_te_m)
prob  = xgb_mc.predict_proba(X_te_m)
f1    = f1_score(y_m_test, pred, average='weighted')
auc   = roc_auc_score(y_m_test, prob, multi_class='ovr', average='weighted')
results_m.append({'model': 'XGBoost', 'F1': f1, 'AUC': auc})
print(f"XGB F1={f1:.4f}  AUC={auc:.4f}")

# ── LightGBM multi-class ──────────────────────────────────────────────────────
lgb_mc = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(np.unique(y_m_train)),
    n_estimators=500, max_depth=7, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    class_weight='balanced',
    random_state=42, n_jobs=-1, verbose=-1
)
lgb_mc.fit(
    X_tr_m, y_m_train,
    eval_set=[(X_te_m, y_m_test)],
    callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=-1)]
)
pred  = lgb_mc.predict(X_te_m)
prob  = lgb_mc.predict_proba(X_te_m)
f1    = f1_score(y_m_test, pred, average='weighted')
auc   = roc_auc_score(y_m_test, prob, multi_class='ovr', average='weighted')
results_m.append({'model': 'LightGBM', 'F1': f1, 'AUC': auc})
print(f"LGB F1={f1:.4f}  AUC={auc:.4f}")

print("\nTask C Summary:")
print(pd.DataFrame(results_m).sort_values('AUC', ascending=False).to_string(index=False))


  TASK C: PRIORITY (MULTI-CLASS)

Class distribution:
severity_enc
1    895
0    664
2    334
3    125
Name: count, dtype: int64

RF  F1=0.4252  AUC=0.6342
XGB F1=0.4063  AUC=0.6445
LGB F1=0.3654  AUC=0.6170

Task C Summary:
       model       F1      AUC
     XGBoost 0.406348 0.644508
RandomForest 0.425220 0.634199
    LightGBM 0.365397 0.616980


In [4]:
# file1_log_transform_features.py
# Technique: log-transform the target + 15 new engineered features
# Expected MAE improvement: 8-15 mins (biggest single gain)

import pandas as pd
import numpy as np
import joblib
import os
import warnings

from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
from xgboost.callback import EarlyStopping
import optuna

df = pd.read_csv('dataset.csv') 

df['closed_datetime']   = pd.to_datetime(df['closed_datetime'],   utc=True, errors='coerce')
df['start_datetime']    = pd.to_datetime(df['start_datetime'],     utc=True, errors='coerce')
df['resolved_datetime'] = pd.to_datetime(df['resolved_datetime'],  utc=True, errors='coerce')
df['end_datetime']      = pd.to_datetime(df['end_datetime'],       utc=True, errors='coerce')
df['modified_datetime'] = pd.to_datetime(df['modified_datetime'],  utc=True, errors='coerce')
df['created_date']      = pd.to_datetime(df['created_date'],       utc=True, errors='coerce')

df['end_ts']        = df['closed_datetime'].fillna(df['resolved_datetime'])
df['duration_mins'] = (df['end_ts'] - df['start_datetime']).dt.total_seconds() / 60

df['response_delay_mins'] = (
    df['start_datetime'] - df['created_date']
).dt.total_seconds().div(60).clip(-60, 1440)

df['update_speed_mins'] = (
    df['modified_datetime'] - df['start_datetime']
).dt.total_seconds().div(60).clip(0, 1440)

df['severity'] = pd.cut(
    df['duration_mins'],
    bins=[0, 30, 90, 240, 1440],
    labels=['low', 'medium', 'high', 'critical'],
    include_lowest=True
)
severity_map = {'low': 0, 'medium': 1, 'high': 2, 'critical': 3}
df['severity_enc'] = df['severity'].map(severity_map)

mask = (
    df['duration_mins'].notna() &
    df['severity_enc'].notna() &
    (df['duration_mins'] > 0) &
    (df['duration_mins'] < 1440) &
    df['priority'].notna() &
    df['address'].notna()
)
df_clean = df[mask].copy()
print(f"Rows: {len(df_clean)}")

for col in ['zone', 'junction', 'gba_identifier', 'veh_type', 'corridor', 'event_cause', 'event_type', 'status']:
    df_clean[col] = df_clean[col].fillna('Unknown')

df_clean['requires_road_closure'] = df_clean['requires_road_closure'].astype(int)

df_clean['hour']         = df_clean['start_datetime'].dt.hour
df_clean['day_of_week']  = df_clean['start_datetime'].dt.dayofweek
df_clean['month']        = df_clean['start_datetime'].dt.month
df_clean['week_of_year'] = df_clean['start_datetime'].dt.isocalendar().week.astype(int)
df_clean['quarter']      = df_clean['start_datetime'].dt.quarter
df_clean['is_weekend']   = df_clean['day_of_week'].isin([5, 6]).astype(int)
df_clean['is_peak_am']   = df_clean['hour'].isin([7, 8, 9]).astype(int)
df_clean['is_peak_pm']   = df_clean['hour'].isin([17, 18, 19]).astype(int)
df_clean['is_peak']      = (df_clean['is_peak_am'] | df_clean['is_peak_pm']).astype(int)
df_clean['is_night']     = df_clean['hour'].isin([22, 23, 0, 1, 2]).astype(int)
df_clean['is_office']    = df_clean['hour'].isin(range(9, 18)).astype(int)
df_clean['is_monday']    = (df_clean['day_of_week'] == 0).astype(int)
df_clean['is_friday']    = (df_clean['day_of_week'] == 4).astype(int)

df_clean['hour_sin']    = np.sin(2 * np.pi * df_clean['hour'] / 24)
df_clean['hour_cos']    = np.cos(2 * np.pi * df_clean['hour'] / 24)
df_clean['dow_sin']     = np.sin(2 * np.pi * df_clean['day_of_week'] / 7)
df_clean['dow_cos']     = np.cos(2 * np.pi * df_clean['day_of_week'] / 7)
df_clean['month_sin']   = np.sin(2 * np.pi * df_clean['month'] / 12)
df_clean['month_cos']   = np.cos(2 * np.pi * df_clean['month'] / 12)

df_clean['lat_bin']          = pd.cut(df_clean['latitude'],  bins=10, labels=False)
df_clean['lon_bin']          = pd.cut(df_clean['longitude'], bins=10, labels=False)
df_clean['has_end_location'] = (df_clean['endlatitude'].fillna(0) != 0).astype(int)

df_clean['geo_span'] = np.sqrt(
    (df_clean['endlatitude'].fillna(df_clean['latitude'])  - df_clean['latitude'])**2 +
    (df_clean['endlongitude'].fillna(df_clean['longitude']) - df_clean['longitude'])**2
)

# Distance from city center (Bengaluru: 12.9716, 77.5946)
df_clean['dist_from_center'] = np.sqrt(
    (df_clean['latitude']  - 12.9716)**2 +
    (df_clean['longitude'] - 77.5946)**2
)

# ─── Label encoding ───────────────────────────────────────────────────────────
LE_COLS = ['event_type', 'event_cause', 'veh_type', 'status']
le_encoders = {}
for col in LE_COLS:
    le = LabelEncoder()
    df_clean[f'{col}_enc'] = le.fit_transform(df_clean[col].astype(str))
    le_encoders[col] = le

priority_map = {'low': 0, 'medium': 1, 'high': 2, 'critical': 3}
df_clean['priority_enc'] = df_clean['priority'].str.lower().map(priority_map).fillna(1).astype(int)

# ─── Target encoding ─────────────────────────────────────────────────────────
for col in ['zone', 'junction', 'corridor', 'gba_identifier', 'event_cause', 'police_station']:
    if col in df_clean.columns:
        mean_map = df_clean.groupby(col)['duration_mins'].mean()
        df_clean[f'{col}_tenc']  = df_clean[col].map(mean_map).fillna(df_clean['duration_mins'].mean())
        df_clean[f'{col}_count'] = df_clean[col].map(df_clean[col].value_counts()).fillna(0)
        df_clean[f'{col}_known'] = (df_clean[col] != 'Unknown').astype(int)

# ─── Interaction features  ───
df_clean['cause_x_peak']       = df_clean['event_cause_enc'] * df_clean['is_peak']
df_clean['cause_x_weekend']    = df_clean['event_cause_enc'] * df_clean['is_weekend']
df_clean['cause_x_closure']    = df_clean['event_cause_enc'] * df_clean['requires_road_closure']
df_clean['hour_x_closure']     = df_clean['hour'] * df_clean['requires_road_closure']
df_clean['zone_x_peak']        = df_clean['zone_tenc'] * df_clean['is_peak']
df_clean['zone_x_cause']       = df_clean['zone_tenc'] * df_clean['event_cause_enc']
df_clean['closure_x_has_end']  = df_clean['requires_road_closure'] * df_clean['has_end_location']
df_clean['priority_x_cause']   = df_clean['priority_enc'] * df_clean['event_cause_enc']
df_clean['dist_x_peak']        = df_clean['dist_from_center'] * df_clean['is_peak']

# ─── Response delay features ──────────────────────────────────────────────────
df_clean['response_delay_mins'] = df_clean['response_delay_mins'].fillna(
    df_clean['response_delay_mins'].median()
)
df_clean['update_speed_mins'] = df_clean['update_speed_mins'].fillna(
    df_clean['update_speed_mins'].median()
)
df_clean['fast_response']     = (df_clean['response_delay_mins'] < 10).astype(int)

# ─── Historical rate features: avg duration by cause at this hour ─────────────
df_clean['cause_hour_mean'] = df_clean.groupby(
    ['event_cause', 'hour']
)['duration_mins'].transform('mean')

df_clean['cause_dow_mean']  = df_clean.groupby(
    ['event_cause', 'day_of_week']
)['duration_mins'].transform('mean')

df_clean['zone_hour_mean']  = df_clean.groupby(
    ['zone', 'hour']
)['duration_mins'].transform('mean')

# ─── Define feature set ───────────────────────────────────────────────────────
FEATURE_COLS = [
    # temporal raw
    'hour', 'day_of_week', 'month', 'week_of_year', 'quarter',
    'is_weekend', 'is_peak_am', 'is_peak_pm', 'is_peak',
    'is_night', 'is_office', 'is_monday', 'is_friday',
    # cyclical
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    # geo
    'latitude', 'longitude', 'lat_bin', 'lon_bin',
    'has_end_location', 'geo_span', 'dist_from_center',
    # event
    'event_type_enc', 'event_cause_enc', 'requires_road_closure',
    'veh_type_enc', 'status_enc', 'priority_enc',
    # target encoded
    'zone_tenc',          'zone_count',          'zone_known',
    'junction_tenc',      'junction_count',       'junction_known',
    'corridor_tenc',      'corridor_count',       'corridor_known',
    'gba_identifier_tenc','gba_identifier_count', 'gba_identifier_known',
    'police_station_tenc','police_station_count', 'police_station_known',
    # interactions
    'cause_x_peak', 'cause_x_weekend', 'cause_x_closure',
    'hour_x_closure', 'zone_x_peak', 'zone_x_cause',
    'closure_x_has_end', 'priority_x_cause', 'dist_x_peak',
    # response speed
    'response_delay_mins', 'update_speed_mins', 'fast_response',
    # historical rates
    'cause_hour_mean', 'cause_dow_mean', 'zone_hour_mean',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_clean.columns]
print(f"Total features: {len(FEATURE_COLS)}")

# ─── LOG-TRANSFORM TARGET ────────────────────────────────────────────────────
# This is the key change: predict log(duration), then exponentiate back
# Compresses the right tail, makes residuals more symmetric
df_clean['log_duration'] = np.log1p(df_clean['duration_mins'])

X     = df_clean[FEATURE_COLS].fillna(-999)
y_log = df_clean['log_duration']
y_raw = df_clean['duration_mins']

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ─── Optuna on log-space target ───────────────────────────────────────────────
def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 300, 1500),
        'max_depth':         trial.suggest_int('max_depth', 4, 10),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'gamma':             trial.suggest_float('gamma', 0.0, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 3.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.5, 5.0),
        'max_delta_step':    trial.suggest_int('max_delta_step', 0, 5),
        'random_state': 42, 'verbosity': 0, 'n_jobs': -1,
    }
    maes = []
    for tr_idx, va_idx in kf.split(X):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y_log.iloc[tr_idx], y_log.iloc[va_idx]
        y_raw_va   = y_raw.iloc[va_idx]

        m = xgb.XGBRegressor(**params)
        m.fit(X_tr, y_tr,
              eval_set=[(X_va, y_va)],
              verbose=False,
            )

        # Evaluate in ORIGINAL space (not log space)
        pred_log = m.predict(X_va)
        pred_raw = np.expm1(pred_log)
        maes.append(mean_absolute_error(y_raw_va, pred_raw))
    return np.mean(maes)

print("Running Optuna (120 trials, log-space target)...")
study = optuna.create_study(direction='minimize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=120, show_progress_bar=True)

best_params = study.best_params
print(f"\nBest OOF MAE (original space): {study.best_value:.2f} mins")
print(f"Best params: {best_params}")

# ─── Final evaluation with CV ────────────────────────────────────────────────
from sklearn.model_selection import cross_val_predict

final_model = xgb.XGBRegressor(**best_params, verbosity=0, n_jobs=-1)

oof_log = cross_val_predict(final_model, X, y_log, cv=kf)
oof_raw = np.expm1(oof_log)

mae  = mean_absolute_error(y_raw, oof_raw)
rmse = np.sqrt(mean_squared_error(y_raw, oof_raw))
r2   = r2_score(y_raw, oof_raw)

print(f"\n{'='*45}")
print(f"  FILE 1 RESULT — Log-transform + interactions")
print(f"  OOF MAE:  {mae:.2f} mins")
print(f"  OOF RMSE: {rmse:.2f} mins")
print(f"  OOF R²:   {r2:.4f}")
print(f"{'='*45}")

# ─── Save ────────────────────────────────────────────────────────────────────
os.makedirs('models_v2', exist_ok=True)
final_model.fit(X, y_log)   # refit on all data
joblib.dump(final_model,  'models_v2/xgb_log_duration.pkl')
joblib.dump(FEATURE_COLS, 'models_v2/feature_cols_v2.pkl')
print("Saved → models_v2/xgb_log_duration.pkl")
print("NOTE: predictions from this model need np.expm1() applied")

[I 2026-06-21 11:11:13,794] A new study created in memory with name: no-name-7d1bda40-01a7-4fbc-8616-5d359001e0ae


Rows: 2523
Total features: 62
Running Optuna (120 trials, log-space target)...


  0%|          | 0/120 [00:00<?, ?it/s]

[I 2026-06-21 11:11:27,336] Trial 0 finished with value: 21.068987503279143 and parameters: {'n_estimators': 749, 'max_depth': 10, 'learning_rate': 0.07441632389160634, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.5780093202212182, 'colsample_bylevel': 0.5779972601681014, 'min_child_weight': 1, 'gamma': 0.8661761457749352, 'reg_alpha': 1.8033450352296265, 'reg_lambda': 3.6863266000822046, 'max_delta_step': 0}. Best is trial 0 with value: 21.068987503279143.
[I 2026-06-21 11:11:55,429] Trial 1 finished with value: 17.28281298530218 and parameters: {'n_estimators': 1464, 'max_depth': 9, 'learning_rate': 0.010943342660062645, 'subsample': 0.6727299868828402, 'colsample_bytree': 0.5917022549267169, 'colsample_bylevel': 0.6521211214797689, 'min_child_weight': 6, 'gamma': 0.43194501864211576, 'reg_alpha': 0.8736874205941257, 'reg_lambda': 3.2533380262507077, 'max_delta_step': 0}. Best is trial 1 with value: 17.28281298530218.
[I 2026-06-21 11:12:10,439] Trial 2 finished with value:

In [6]:
import pandas as pd
import numpy as np
import joblib, os, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Datetime + duration (same as before) ──────────────────────────────────────
for col in ['closed_datetime','start_datetime','resolved_datetime',
            'modified_datetime','created_date']:
    df[col] = pd.to_datetime(df[col], utc=True, errors='coerce')

df['end_ts']        = df['closed_datetime'].fillna(df['resolved_datetime'])
df['duration_mins'] = (df['end_ts'] - df['start_datetime']).dt.total_seconds() / 60
df['severity']      = pd.cut(df['duration_mins'],
                              bins=[0,30,90,240,1440],
                              labels=['low','medium','high','critical'],
                              include_lowest=True)
df['severity_enc']  = df['severity'].map({'low':0,'medium':1,'high':2,'critical':3})

mask = (df['duration_mins'].notna() & df['severity_enc'].notna() &
        (df['duration_mins'] > 0) & (df['duration_mins'] < 1440) &
        df['priority'].notna() & df['address'].notna())
df_clean = df[mask].copy().reset_index(drop=True)
print(f"Rows: {len(df_clean)}")

# ── Fill + standard features ──────────────────────────────────────────────────
for col in ['zone','junction','gba_identifier','veh_type',
            'corridor','event_cause','event_type','status']:
    df_clean[col] = df_clean[col].fillna('Unknown')

df_clean['requires_road_closure'] = df_clean['requires_road_closure'].astype(int)
df_clean['response_delay_mins']   = (
    df_clean['start_datetime'] - df_clean['created_date']
).dt.total_seconds().div(60).clip(-60, 1440).fillna(5.0)
df_clean['update_speed_mins']     = (
    df_clean['modified_datetime'] - df_clean['start_datetime']
).dt.total_seconds().div(60).clip(0, 1440).fillna(10.0)

df_clean['hour']         = df_clean['start_datetime'].dt.hour
df_clean['day_of_week']  = df_clean['start_datetime'].dt.dayofweek
df_clean['month']        = df_clean['start_datetime'].dt.month
df_clean['week_of_year'] = df_clean['start_datetime'].dt.isocalendar().week.astype(int)
df_clean['quarter']      = df_clean['start_datetime'].dt.quarter
df_clean['is_weekend']   = df_clean['day_of_week'].isin([5,6]).astype(int)
df_clean['is_peak']      = df_clean['hour'].isin([7,8,9,17,18,19]).astype(int)
df_clean['is_peak_am']   = df_clean['hour'].isin([7,8,9]).astype(int)
df_clean['is_peak_pm']   = df_clean['hour'].isin([17,18,19]).astype(int)
df_clean['is_night']     = df_clean['hour'].isin([22,23,0,1,2]).astype(int)
df_clean['is_office']    = df_clean['hour'].isin(range(9,18)).astype(int)
df_clean['is_monday']    = (df_clean['day_of_week']==0).astype(int)
df_clean['is_friday']    = (df_clean['day_of_week']==4).astype(int)

df_clean['hour_sin']  = np.sin(2*np.pi*df_clean['hour']/24)
df_clean['hour_cos']  = np.cos(2*np.pi*df_clean['hour']/24)
df_clean['dow_sin']   = np.sin(2*np.pi*df_clean['day_of_week']/7)
df_clean['dow_cos']   = np.cos(2*np.pi*df_clean['day_of_week']/7)
df_clean['month_sin'] = np.sin(2*np.pi*df_clean['month']/12)
df_clean['month_cos'] = np.cos(2*np.pi*df_clean['month']/12)

df_clean['lat_bin']          = pd.cut(df_clean['latitude'],  bins=8, labels=False)
df_clean['lon_bin']          = pd.cut(df_clean['longitude'], bins=8, labels=False)
df_clean['has_end_location'] = (df_clean['endlatitude'].fillna(0)!=0).astype(int)
df_clean['geo_span']         = np.sqrt(
    (df_clean['endlatitude'].fillna(df_clean['latitude'])-df_clean['latitude'])**2 +
    (df_clean['endlongitude'].fillna(df_clean['longitude'])-df_clean['longitude'])**2)
df_clean['dist_from_center'] = np.sqrt(
    (df_clean['latitude']-12.9716)**2+(df_clean['longitude']-77.5946)**2)

LE_COLS = ['event_type','event_cause','veh_type','status']
le_encoders = {}
for col in LE_COLS:
    le = LabelEncoder()
    df_clean[f'{col}_enc'] = le.fit_transform(df_clean[col].astype(str))
    le_encoders[col] = le

df_clean['priority_enc'] = df_clean['priority'].str.lower().map(
    {'low':0,'medium':1,'high':2,'critical':3}).fillna(1).astype(int)
df_clean['fast_response'] = (df_clean['response_delay_mins'] < 10).astype(int)

# ── Target encoding: also must be done fold-by-fold to avoid leakage ─────────
# We'll compute these inside the CV loop below.
# For now, just precompute a GLOBAL version for the final refit only.
TENC_COLS = ['zone','junction','corridor','gba_identifier','police_station']
global_tenc = {}
for col in TENC_COLS:
    if col in df_clean.columns:
        m = df_clean.groupby(col)['duration_mins'].mean()
        global_tenc[col] = m
        df_clean[f'{col}_tenc_global']  = df_clean[col].map(m).fillna(df_clean['duration_mins'].mean())
        df_clean[f'{col}_count']        = df_clean[col].map(df_clean[col].value_counts()).fillna(0)
        df_clean[f'{col}_known']        = (df_clean[col] != 'Unknown').astype(int)

# Interaction features that do NOT use the target
df_clean['cause_x_peak']      = df_clean['event_cause_enc'] * df_clean['is_peak']
df_clean['cause_x_weekend']   = df_clean['event_cause_enc'] * df_clean['is_weekend']
df_clean['cause_x_closure']   = df_clean['event_cause_enc'] * df_clean['requires_road_closure']
df_clean['hour_x_closure']    = df_clean['hour'] * df_clean['requires_road_closure']
df_clean['closure_x_has_end'] = df_clean['requires_road_closure'] * df_clean['has_end_location']
df_clean['priority_x_cause']  = df_clean['priority_enc'] * df_clean['event_cause_enc']
df_clean['dist_x_peak']       = df_clean['dist_from_center'] * df_clean['is_peak']

# ── Feature list (NO historical rate features here — added inside CV) ─────────
BASE_FEATURES = [
    'hour','day_of_week','month','week_of_year','quarter',
    'is_weekend','is_peak','is_peak_am','is_peak_pm',
    'is_night','is_office','is_monday','is_friday',
    'hour_sin','hour_cos','dow_sin','dow_cos','month_sin','month_cos',
    'latitude','longitude','lat_bin','lon_bin',
    'has_end_location','geo_span','dist_from_center',
    'event_type_enc','event_cause_enc','requires_road_closure',
    'veh_type_enc','status_enc','priority_enc',
    'zone_tenc_global','zone_count','zone_known',
    'junction_tenc_global','junction_count','junction_known',
    'corridor_tenc_global','corridor_count','corridor_known',
    'cause_x_peak','cause_x_weekend','cause_x_closure',
    'hour_x_closure','closure_x_has_end','priority_x_cause','dist_x_peak',
    'response_delay_mins','update_speed_mins','fast_response',
]
BASE_FEATURES = [c for c in BASE_FEATURES if c in df_clean.columns]

# ── Log target ────────────────────────────────────────────────────────────────
df_clean['log_duration'] = np.log1p(df_clean['duration_mins'])
y_raw = df_clean['duration_mins'].values
y_log = df_clean['log_duration'].values
n     = len(df_clean)

# ════════════════════════════════════════════════════════════════════════════
#  PROPER CV: historical rate features built on training fold only
# ════════════════════════════════════════════════════════════════════════════
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def add_rate_features_proper(df_tr, df_va, group_cols, target='duration_mins'):
    """
    Compute group mean on training fold only.
    Apply to validation fold with fallback to training global mean.
    Returns (X_tr_extra, X_va_extra) as numpy arrays — two columns per group.
    """
    global_mean = df_tr[target].mean()
    tr_cols, va_cols = [], []

    for gcols in group_cols:
        key = '_x_'.join(gcols)
        # leave-one-out mean for training rows (avoids self-leakage)
        total_sum   = df_tr[target].sum()
        total_count = len(df_tr)

        group_sum   = df_tr.groupby(gcols)[target].sum()
        group_count = df_tr.groupby(gcols)[target].count()

        def loo_mean(row):
            k = tuple(row[c] for c in gcols) if len(gcols)>1 else row[gcols[0]]
            gs = group_sum.get(k, 0)
            gc = group_count.get(k, 0)
            if gc <= 1:
                return global_mean
            return (gs - row[target]) / (gc - 1)

        tr_mean = df_tr.apply(loo_mean, axis=1).values

        # For validation: use training group mean (no leakage possible)
        def va_lookup(row):
            k = tuple(row[c] for c in gcols) if len(gcols)>1 else row[gcols[0]]
            gs = group_sum.get(k, 0)
            gc = group_count.get(k, 0)
            return gs/gc if gc > 0 else global_mean

        va_mean = df_va.apply(va_lookup, axis=1).values

        tr_cols.append(tr_mean)
        va_cols.append(va_mean)

    return np.column_stack(tr_cols), np.column_stack(va_cols)

RATE_GROUPS = [
    ['event_cause', 'hour'],
    ['event_cause', 'day_of_week'],
    ['zone',        'hour'],
    ['event_cause', 'is_peak'],
]

# Quick Optuna search first on BASE_FEATURES only (fast, no per-fold overhead)
print("Step 1: Optuna hyperparameter search on base features...")

X_base = df_clean[BASE_FEATURES].fillna(-999).values

def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 300, 1200),
        'max_depth':         trial.suggest_int('max_depth', 4, 10),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'gamma':             trial.suggest_float('gamma', 0.0, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 3.0),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.5, 5.0),
        'random_state': 42, 'verbosity': 0, 'n_jobs': -1,
    }
    maes = []
    for tr_idx, va_idx in kf.split(X_base):
        m = xgb.XGBRegressor(**params)
        m.fit(X_base[tr_idx], y_log[tr_idx],
              eval_set=[(X_base[va_idx], y_log[va_idx])],
              verbose=False)
        pred = np.expm1(m.predict(X_base[va_idx]))
        maes.append(mean_absolute_error(y_raw[va_idx], pred))
    return np.mean(maes)

study = optuna.create_study(direction='minimize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)
best_params = study.best_params
print(f"Optuna base MAE: {study.best_value:.2f} mins  ← this is your real baseline")

# ── Full OOF with proper rate features ───────────────────────────────────────
print("\nStep 2: OOF evaluation WITH properly-built rate features...")

oof_preds = np.zeros(n)

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_base)):
    df_tr = df_clean.iloc[tr_idx].copy()
    df_va = df_clean.iloc[va_idx].copy()

    X_tr_base = df_clean[BASE_FEATURES].fillna(-999).iloc[tr_idx].values
    X_va_base = df_clean[BASE_FEATURES].fillna(-999).iloc[va_idx].values

    # Build rate features on training fold only
    X_tr_rate, X_va_rate = add_rate_features_proper(df_tr, df_va, RATE_GROUPS)

    X_tr = np.hstack([X_tr_base, X_tr_rate])
    X_va = np.hstack([X_va_base, X_va_rate])

    m = xgb.XGBRegressor(**best_params, verbosity=0, n_jobs=-1, random_state=42)
    m.fit(X_tr, y_log[tr_idx])
    oof_preds[va_idx] = np.expm1(m.predict(X_va))

    fold_mae = mean_absolute_error(y_raw[va_idx], oof_preds[va_idx])
    print(f"  Fold {fold+1}: MAE = {fold_mae:.2f} mins")

mae  = mean_absolute_error(y_raw, oof_preds)
rmse = np.sqrt(mean_squared_error(y_raw, oof_preds))
r2   = r2_score(y_raw, oof_preds)

print(f"\n{'='*50}")
print(f"  FILE 1 FIXED — Log-transform + leak-free rate features")
print(f"  OOF MAE:  {mae:.2f} mins  (this is trustworthy)")
print(f"  OOF RMSE: {rmse:.2f} mins")
print(f"  OOF R²:   {r2:.4f}")
print(f"{'='*50}")

# ── Save final model (refit on all data with global tenc) ────────────────────
# Rate features for full refit: use leave-one-out on entire dataset
print("\nRefitting on full data for saving...")

def loo_rate_full(df_full, gcols, target='duration_mins'):
    global_mean = df_full[target].mean()
    group_sum   = df_full.groupby(gcols)[target].sum()
    group_count = df_full.groupby(gcols)[target].count()
    def loo(row):
        k = tuple(row[c] for c in gcols) if len(gcols)>1 else row[gcols[0]]
        gs = group_sum.get(k, 0)
        gc = group_count.get(k, 0)
        return (gs - row[target]) / (gc - 1) if gc > 1 else global_mean
    return df_full.apply(loo, axis=1).values

rate_cols_full = np.column_stack([
    loo_rate_full(df_clean, g) for g in RATE_GROUPS
])
X_full = np.hstack([
    df_clean[BASE_FEATURES].fillna(-999).values,
    rate_cols_full
])

final_model = xgb.XGBRegressor(**best_params, verbosity=0, n_jobs=-1, random_state=42)
final_model.fit(X_full, y_log)

os.makedirs('models_v2', exist_ok=True)
joblib.dump(final_model,   'models_v2/xgb_log_fixed.pkl')
joblib.dump(BASE_FEATURES, 'models_v2/base_features_fixed.pkl')
joblib.dump(RATE_GROUPS,   'models_v2/rate_groups.pkl')
joblib.dump(global_tenc,   'models_v2/global_tenc_fixed.pkl')
joblib.dump(le_encoders,   'models_v2/label_encoders_fixed.pkl')
print("Saved → models_v2/xgb_log_fixed.pkl")
print("\nNOTE: predictions need np.expm1() applied to model output")
print("NOTE: rate features must be approximated at inference time using")
print("      global group means from global_tenc_fixed.pkl")

Rows: 2523
Step 1: Optuna hyperparameter search on base features...


  0%|          | 0/100 [00:00<?, ?it/s]

Optuna base MAE: 6.77 mins  ← this is your real baseline

Step 2: OOF evaluation WITH properly-built rate features...
  Fold 1: MAE = 11.94 mins
  Fold 2: MAE = 8.21 mins
  Fold 3: MAE = 13.14 mins
  Fold 4: MAE = 12.44 mins
  Fold 5: MAE = 10.97 mins

  FILE 1 FIXED — Log-transform + leak-free rate features
  OOF MAE:  11.34 mins  (this is trustworthy)
  OOF RMSE: 49.53 mins
  OOF R²:   0.9415

Refitting on full data for saving...
Saved → models_v2/xgb_log_fixed.pkl

NOTE: predictions need np.expm1() applied to model output
NOTE: rate features must be approximated at inference time using
      global group means from global_tenc_fixed.pkl


In [7]:
import joblib
le = joblib.load('models_v2/label_encoders_fixed.pkl')
print(le['status'].classes_)
# ['closed', 'resolved']
# closed=0, resolved=1

# Check duration distribution by status in training data
print(df_clean.groupby('status')['duration_mins'].describe())

['closed' 'resolved']
           count       mean         std       min        25%        50%  \
status                                                                    
closed    2458.0  99.046653  206.502399  0.696538  22.109145  45.657271   
resolved    65.0  78.112089  125.016156  3.311959  32.965140  58.346161   

                75%          max  
status                            
closed    85.276225  1437.053099  
resolved  83.154634   980.117226  


In [8]:
import joblib
import numpy as np

model = joblib.load('models_v2/xgb_log_fixed.pkl')
feat  = joblib.load('models_v2/base_features_fixed.pkl')

# Feature importances
importances = model.feature_importances_
pairs = sorted(zip(feat + ['cause_hour_rate','cause_dow_rate','zone_hour_rate','cause_peak_rate'],
                   importances), key=lambda x: x[1], reverse=True)

print("Top 15 feature importances:")
for name, imp in pairs[:15]:
    print(f"  {name:30s}  {imp:.4f}")

print(f"\nBottom 10 (near-zero importance):")
for name, imp in pairs[-10:]:
    print(f"  {name:30s}  {imp:.4f}")

# Check what the model predicts for extreme inputs
print("\n--- Sanity checks ---")
# All zeros
zero_X = np.zeros((1, 55), dtype=np.float32)
print(f"All zeros input:     {np.expm1(model.predict(zero_X)[0]):.1f} mins")

# Max road closure + peak hour
test_X = np.zeros((1, 55), dtype=np.float32)
feat_list = feat + ['cause_hour_rate','cause_dow_rate','zone_hour_rate','cause_peak_rate']
feat_idx = {f: i for i, f in enumerate(feat_list)}

test_X[0, feat_idx['requires_road_closure']] = 1
test_X[0, feat_idx['is_peak']]               = 1
test_X[0, feat_idx['zone_tenc_global']]       = 200.0
test_X[0, feat_idx['cause_hour_rate']]        = 300.0
print(f"Road closure+peak+high zone: {np.expm1(model.predict(test_X)[0]):.1f} mins")

# Check log target distribution
print(f"\nTraining log_duration stats:")
log_dur = np.log1p(df_clean['duration_mins'])
print(f"  mean:  {log_dur.mean():.4f}  → {np.expm1(log_dur.mean()):.1f} mins")
print(f"  std:   {log_dur.std():.4f}")
print(f"  min:   {log_dur.min():.4f}  → {np.expm1(log_dur.min()):.1f} mins")
print(f"  max:   {log_dur.max():.4f}  → {np.expm1(log_dur.max()):.1f} mins")
print(f"  median:{np.median(log_dur):.4f}  → {np.expm1(np.median(log_dur)):.1f} mins")

Top 15 feature importances:
  update_speed_mins               0.4430
  cause_dow_rate                  0.0916
  cause_hour_rate                 0.0736
  cause_peak_rate                 0.0597
  corridor_tenc_global            0.0416
  veh_type_enc                    0.0388
  dist_from_center                0.0306
  event_cause_enc                 0.0262
  junction_tenc_global            0.0241
  is_peak                         0.0232
  junction_count                  0.0140
  latitude                        0.0113
  month                           0.0104
  hour_sin                        0.0101
  zone_hour_rate                  0.0100

Bottom 10 (near-zero importance):
  geo_span                        0.0000
  requires_road_closure           0.0000
  zone_count                      0.0000
  zone_known                      0.0000
  junction_known                  0.0000
  corridor_known                  0.0000
  cause_x_closure                 0.0000
  closure_x_has_end               0